#Practice with Delta Tables

## Task: Upload the people.csv file to my learning volume and list it
Note: Let´s see 3 different ways to list files in a path

In [0]:
%fs ls '/Volumes/workspace/default/learning'

In [0]:
# Using Python code instead
display(dbutils.fs.ls('/Volumes/workspace/default/learning'))

In [0]:
%sql
-- Using SQL stament instead
LIST '/Volumes/workspace/default/learning'

### Task: Read a CSV file with a tabular format using Databricks SQL statement 

In [0]:
%sql
select * from csv.`/Volumes/workspace/default/learning/people.csv`;

## Task: Create a Delta Table from the CSV file

In [0]:
# Step 1: Create a DataFrame from the CSV file
df = spark.read.csv("/Volumes/workspace/default/learning/people.csv", header=True, inferSchema=True)
display(df)

# Step 2: Create a Delta Table from the DataFrame
df.write.format("delta").saveAsTable("workspace.default.people_table")

##Task: SQL statement to select * from the Delta Table

In [0]:
%sql
select * from people_table;

##Task:
- Insert 2 new people
- Update the Alice´s salary to 50500


In [0]:
%sql
insert into people_table
values 
  (11, 'Carlos', 42, 50000),
  (12, 'Lenka', 40, 60000);

  update people_table
  set salary = 70000
  where id = 1;


In [0]:
%sql
select * from people_table order by id;


##View the table history

In [0]:
%sql
describe history people_table;

#Practice Delta Lake LakeFlow Connect techniques

## CTAS - Create Table As

Databricks SQL syntax: Call the read_files() function 

In [0]:
%sql
-- Let´s do a previous SELECT.read_file() just to see the table
select * from read_files(
  '/Volumes/workspace/default/learning/customer.csv', -- Si hubiera más CSV files dentro de learning, se leen todos
  format => 'csv'
) limit 10;

-- OBS! Notice the column '_rescued_datsa' which is automatically included to capture any data that does not match the infered schema

In [0]:
%sql
-- Drop table if it exists
drop table if exists customers_ctas;

-- Create the Delta Table
create table customers_ctas
as
select customer_id, first_name, last_name, date_of_birth, gender
from read_files(
  '/Volumes/workspace/default/learning/customer.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

select * from customers_ctas limit 10;

-- show tables;

##Interesting exercise to handle data from the _rescued_data

In [0]:
%sql
-- Task: Create the Delta Table explictly indicating the schema
create table customer_ctas_schema
as
select *
from read_files(
  '/Volumes/workspace/default/learning/customer.csv',
  format => 'csv',
  header => true,
  schema => '''
    customer_id int,
    first_name string,
    last_name string,
    date_of_birth date,
    gender string''',
  rescueddatacolumn => '_rescued_data'
);


In [0]:
%sql
select * from customer_ctas_schema;
-- OBS! Al indicar ahora el schema, concretamente date_of_birth no sabe qué poner y se guarda en _rescued_data
-- Cómo rescatar estos datos? Mira the next cell...

In [0]:
%sql
-- How to handle missing data that exist in _rescued_data column?
-- OBS! Los valores the _rescued_data están en JSON format. Vamos a ver cómo se accede a ellos
select
  to_date( _rescued_data:date_of_birth, 'dd-MM-yyyy') as date_of_birth
from customer_ctas_schema
limit 10;

In [0]:
%sql
-- Now let´s bring the corrected data back to the table
UPDATE customer_ctas_schema
SET date_of_birth = to_date(_rescued_data:date_of_birth, 'dd-MM-yyyy')
WHERE date_of_birth IS NULL AND _rescued_data:date_of_birth IS NOT NULL;

In [0]:
%sql
select * from customer_ctas_schema;

In [0]:
%sql
describe table extended customers_ctas;
-- OBS! Note that is says Type: MANAGED

Python Syntax: Call spark.read() method to ingest the file and write() method to create the Delta Table

In [0]:
# Step 1: Create a DataFrame from the CSV file
df = spark.read.csv("/Volumes/workspace/default/learning/customer.csv", header=True, inferSchema=True)
display(df)
# OBS! Creating the DataFrame in Python does not create by default the _rescued_data column, unless explicitely indicated as one of the read options

# Step 2: Create a Delta Table from the DataFrame
df.write.format("delta").saveAsTable("workspace.default.customer_table")

# Step 3: Read and view the table
customer_table = spark.read.table("workspace.default.customer_table")
display(customer_table)
#customer_table.display()

##Upload UI technique
Steps:
- Open the catalog browser on the schema level
- Follow the steps to create a table

##COPY INTO
The demo video says that "COPY INTO" is now legacy and the new way to do incremental load is by using Auto Loader instead

In [0]:
%sql
-- Create empty table with schema
create table if not exists customers_copyinto(
  customer_id int,
  first_name string,
  last_name string,
  date_of_birth date,
  gender string);

-- Populate from a file using copy into
copy into customers_copyinto
from '/Volumes/workspace/default/learning/customer.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');
-- COPY_OPTIONS can be either mergeSchema (=> schema evoltution) or force (=> schema override)
-- COPY_OPTIONS ('mergeSchema' = 'true');

  -- OBS! Lo interesante de este método es que permite hacer incremental loads,
  --      simplemente anadiendo nuevos ficheros al volumen
  --      y ejecutando el mismo copy into.e
  -- En este caso, from 'path del volumen (sin indicar fichero)'

#Data Transformation Overview

##Bronze layer
Raw data ingestion

In [0]:
%sql
create table if not exists employees_bronze (
  id int,
  name string,
  country string,
  role string);

copy into employees_bronze
from '/Volumes/workspace/default/learning/employees_dataset/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')

In [0]:
%sql
select * from employees_bronze;

###Additionally, we may want to enrich bronze table with some metadata for traceability

In [0]:
%sql
select 
  *,
  _metadata.file_path as file_path,
  _metadata.file_modification_time as file_modification_time,
  current_timestamp() as current_timestamp
from employees_bronze;
-- OBS! Las columnas con metadatos se autopopulan

-- OBS! Este es el select block que vamos a usar para crear la tabla, en caso de no haberla creado antes

##Silver layer
Basic transformations:
- Select id, name and country from bronze layer
- Convert role to uppercase
- Add 2 new cols: current_timestamp and current_date 

In [0]:
%sql
create table if not exists employees_silver as
select
  id,
  name,
  country,
  upper(role) as role,
  current_timestamp() as current_timestamp,
  date(current_timestamp) as current_date
from employees_bronze;

In [0]:
%sql
select * from employees_silver;

##Gold layer
Aggregate the silver table to create the gold table.
Steps:
- Create a temp_view that aggregates the total number of employees by role
-  Create a table total_roles_gold (Data Mart)

In [0]:
%sql
-- Step 1
create view temp_view as
select
  role,
  count(*) as total
from employees_silver
group by role;

select * from temp_view;
    
-- Step 2
create table if not exists total_roles_gold (
  role string,
  total int
);

insert into total_roles_gold
select * from temp_view;

select * from total_roles_gold;